<a href="https://colab.research.google.com/github/shreyansh8h/hello-world/blob/master/crewai_end_to_end_agent_openai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CrewAI Concierge Agent with OpenAI (Lightweight Model)

This notebook shows a complete CrewAI workflow using a concierge-style agent and tools.

Audience:
- Developers who want a practical non-research CrewAI project.

Prerequisites:
- Python environment with internet access
- `OPENAI_API_KEY` set in your environment

Learning goals:
- Build custom tools in CrewAI
- Create concierge and operations agents
- Run an end-to-end workflow with a lightweight OpenAI model


## Workflow

1. Install dependencies
2. Configure model + key
3. Define concierge tools
4. Build agents and tasks
5. Execute the crew
6. Save output


In [ ]:
%pip install -qU crewai openai python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.7/360.7 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.

In [ ]:
import os
from datetime import datetime
from pathlib import Path

from crewai import Agent, Crew, LLM, Process, Task
from crewai.tools import tool

# Used to securely store your API key
from google.colab import userdata

# Set your OpenAI API key here. Replace "YOUR_API_KEY_HERE" with your actual key.
# For Colab, you might also consider using `from google.colab import userdata`
# and `os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')`
# or loading from a .env file using `python-dotenv`.
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Set OPENAI_API_KEY before running this notebook.")

llm = LLM(
    model=f"openai/{OPENAI_MODEL}",
    temperature=0.2,
)

print(f"Using model: {OPENAI_MODEL}")

Using model: gpt-4o-mini


In [ ]:
@tool("City Activity Lookup")
def city_activity_lookup(city: str) -> str:
    """Return curated activity ideas for a city."""
    activities = {
        "bengaluru": ["Cubbon Park walk", "Lalbagh visit", "Indiranagar cafe trail"],
        "mumbai": ["Marine Drive sunset", "Colaba street food", "Kala Ghoda art walk"],
        "delhi": ["Lodhi Garden", "India Gate evening", "Dilli Haat food + crafts"],
        "hyderabad": ["Charminar + Laad Bazaar", "Tank Bund evening", "Biryani trail"],
    }
    key = city.strip().lower()
    picks = activities.get(key, ["Local city center walk", "Popular food district", "Top-rated museum"])
    return " | ".join(picks)

@tool("Budget Checker")
def budget_checker(total_budget_usd: float, planned_spend_usd: float) -> str:
    """Compare planned spend against a total budget and return a short verdict."""
    diff = round(total_budget_usd - planned_spend_usd, 2)
    if diff >= 0:
        return f"Within budget. Remaining: ${diff}"
    return f"Over budget by ${abs(diff)}"

@tool("Packing List Generator")
def packing_list_generator(weather: str, trip_days: int) -> str:
    """Generate a quick packing list from weather and trip length."""
    base = ["Phone charger", "ID/Passport", "Toiletries", "1 pair comfortable shoes"]
    weather_key = weather.strip().lower()
    if "rain" in weather_key:
        base += ["Umbrella", "Light rain jacket"]
    elif "hot" in weather_key:
        base += ["Sunscreen", "Cap", "Reusable water bottle"]
    elif "cold" in weather_key:
        base += ["Jacket", "Warm socks"]
    outfits = max(2, trip_days)
    base.append(f"{outfits} outfit changes")
    return " | ".join(base)


In [ ]:
concierge_agent = Agent(
    role="Travel Concierge",
    goal="Create personalized trip plans that match preferences and budget.",
    backstory="You are a friendly concierge who combines practical planning with local flavor.",
    llm=llm,
    tools=[city_activity_lookup, budget_checker, packing_list_generator],
    verbose=True,
    allow_delegation=False,
)

operations_agent = Agent(
    role="Operations Coordinator",
    goal="Convert concierge notes into a final, structured client-ready plan.",
    backstory="You produce clear itineraries with timelines, budgets, and checklists.",
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

concierge_task = Task(
    description=(
        "Create a trip concept for {city} for {trip_days} days for {traveler_type}. "
        "You must use at least two tools. Include activities, estimated spend, and a packing list. "
        "Respect a total budget of ${budget_usd}."
    ),
    expected_output="A draft trip concept with activities, spend estimate, and packing guidance.",
    agent=concierge_agent,
)

operations_task = Task(
    description=(
        "Transform the concierge output into a polished markdown brief. "
        "Include sections: Overview, Day-by-day plan, Budget summary table, and Final checklist."
    ),
    expected_output="A client-ready markdown itinerary with structure and clear next actions.",
    agent=operations_agent,
)

crew = Crew(
    agents=[concierge_agent, operations_agent],
    tasks=[concierge_task, operations_task],
    process=Process.sequential,
    verbose=True,
)

In [ ]:
inputs = {
    "city": "Bengaluru",
    "trip_days": 3,
    "traveler_type": "first-time business traveler who also likes local food",
    "budget_usd": 220,
}

result = crew.kickoff(inputs=inputs)

final_text = result.raw if hasattr(result, "raw") else str(result)
print(final_text)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: ea1e7062-876d-4b76-9965-0b1ebb658334                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Create a trip concept for Bengaluru for 3 days for first-time business traveler who also likes local     │
│  food. You must use at least two tools. Include activities, estimated spend, and a packing list. Respect a      │
│  total budget of $220.                                                                                          │
│  ID: 89609fe5-2469-4da4-9483-ee90d5b394a6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Travel Concierge                                                                                        │
│                                                                                                                 │
│  Task: Create a trip concept for Bengaluru for 3 days for first-time business traveler who also likes local     │
│  food. You must use at least two tools. Include activities, estimated spend, and a packing list. Respect a      │
│  total budget of $220.                                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: city_activity_lookup                                                                                     │
│  Args: {'city': 'Bengaluru'}                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool city_activity_lookup executed with result: Cubbon Park walk | Lalbagh visit | Indiranagar cafe trail...
Tool packing_list_generator executed with result: Phone charger | ID/Passport | Toiletries | 1 pair comfortable shoes | 3 outfit changes...


...

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Transform the concierge output into a polished markdown brief. Include sections: Overview, Day-by-day    │
│  plan, Budget summary table, and Final checklist.                                                               │
│  ID: 3e5a1800-8742-4c41-9029-ad6c06b7b569                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Operations Coordinator                                                                                  │
│                                                                                                                 │
│  Task: Transform the concierge output into a polished markdown brief. Include sections: Overview, Day-by-day    │
│  plan, Budget summary table, and Final checklist.                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Operations Coordinator                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```markdown                                                                                                    │
│  # Bengaluru Trip Itinerary (3 Days)                                                                            │
│                                                                                                                 │
│  ## Overview                                                                                                    │
│  Welcome to your 3-day adventure in Bengaluru! This itinerary is designed to immerse you in the local culture,  │
│  cuisine, and natural beauty of the city. From exploring lush gardens to indulging in local delicacies, you     │
│  will experience the essence of Bengaluru.                                                                      │
│                                                                                                                 │
│  ## Day-by-Day Plan                                                                                             │
│                                                                                                                 │
│  ### Day 1: Arrival and Local Exploration                                                                       │
│  - **Morning:**                                                                                                 │
│    - Arrive in Bengaluru and check into your hotel.                                                             │
│  - **Activity:**                                                                                                │
│    - Visit **Cubbon Park** for a refreshing walk. Enjoy the greenery and local flora.                           │
│  - **Lunch:**                                                                                                   │
│    - Try local cuisine at a nearby restaurant (Estimated Spend: $10).                                           │
│  - **Afternoon:**                                                                                               │
│    - Explore **Lalbagh Botanical Garden**, famous for its diverse plant species and beautiful landscapes.       │
│  - **Dinner:**                                                                                                  │
│    - Enjoy a meal at a local eatery (Estimated Spend: $15).                                                     │
│                                                                                                                 │
│  ### Day 2: Culture and Cafes                                                                                   │
│  - **Morning:**                                                                                                 │
│    - Visit the **Bangalore Palace** to learn about the city's history.                                          │
│  - **Lunch:**                                                                                                   │
│    - Head to **Indiranagar** for a cafe trail, sampling local snacks and coffee (Estimated Spend: $15).         │
│  - **Afternoon:**                                                                                               │
│    - Visit local markets for shopping and cultural immersion.                                                   │
│  - **Dinner:**                                         

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Transform the concierge output into a polished markdown brief. Include sections: Overview, Day-by-day    │
│  plan, Budget summary table, and Final checklist.                                                               │
│  Agent: Operations Coordinator                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

```markdown
# Bengaluru Trip Itinerary (3 Days)

## Overview
Welcome to your 3-day adventure in Bengaluru! This itinerary is designed to immerse you in the local culture, cuisine, and natural beauty of the city. From exploring lush gardens to indulging in local delicacies, you will experience the essence of Bengaluru.

## Day-by-Day Plan

### Day 1: Arrival and Local Exploration
- **Morning:**
  - Arrive in Bengaluru and check into your hotel.
- **Activity:**
  - Visit **Cubbon Park** for a refreshing walk. Enjoy the greenery and local flora.
- **Lunch:**
  - Try local cuisine at a nearby restaurant (Estimated Spend: $10).
- **Afternoon:**
  - Explore **Lalbagh Botanical Garden**, famous for its diverse plant species and beautiful landscapes.
- **Dinner:**
  - Enjoy a meal at a local eatery (Estimated Spend: $15).

### Day 2: Culture and Cafes
- **Morning:**
  - Visit the **Bangalore Palace** to learn about the city's history.
- **Lunch:**
  - Head to **Indiranagar** for a cafe trail, 

In [ ]:
out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_file = out_dir / f"crewai_concierge_agent_{timestamp}.md"
out_file.write_text(final_text, encoding="utf-8")
out_file

PosixPath('outputs/crewai_concierge_agent_20260426_131019.md')

## Next Experiments

- Switch model by setting `OPENAI_MODEL` (for example: `gpt-4o-mini`).
- Replace the toy tools with real APIs (maps, weather, booking).
- Add a third QA agent that validates cost and timeline consistency.
- Change `Process.sequential` to test different coordination strategies.
